# 🧬 Lab-Specific Physics for Better VLAs — Threading Mechanisms in MuJoCo

### Reproducing the core idea of **AutoBio** ([arXiv:2505.14030](https://arxiv.org/abs/2505.14030)) on a single free GPU

**Subtitle.** Why a *signed-distance-function (SDF) helix* contact plugin lets MuJoCo
simulate a screw thread faithfully — and why that physical fidelity matters for
**Vision-Language-Action (VLA)** policies that have to *screw a cap onto a tube*.

| | |
|---|---|
| **Hardware target** | Google Colab **free tier (T4, 16 GB)** — also runs CPU-only, just slower |
| **Core frameworks** | `mujoco==3.3.0` (+ the AutoBio `mjlab.sdf.thread` plugin), `PyTorch`, `numpy`, `matplotlib`, `imageio` |
| **Prerequisites** | Basic rigid-body simulation, contact dynamics, and PyTorch behaviour cloning |
| **Runtime** | ~20–35 min end-to-end on a T4 (most of it is demo collection + tiny-policy training) |

> 🧪 **For research and educational purposes only.** This notebook is a *didactic
> reconstruction* of one idea from AutoBio, not the official benchmark. The small
> policy here is **not** π₀ or RDT (the paper's models need ~1,500 GPU-hours on
> 80 GB H800s); it is a deliberately tiny stand-in so the whole pipeline fits on a T4.

## 1 · What you will learn

The thesis of AutoBio we test here:

> **Lab-specific physics improves the policies you can learn from a simulator.**
> A screw thread cannot be modelled faithfully by default rigid-body contact;
> AutoBio adds a helix-SDF contact plugin that makes threading (including
> *self-locking*) emerge physically. We show that demonstrations from the
> *faithful* simulator produce a better cap-screwing policy than demonstrations
> from a physics-naïve one.

| # | Milestone | What we measure |
|---|-----------|-----------------|
| 1 | Environment & plugin setup | load the x86-64 `mjlab.sdf.thread` plugin on Colab |
| 2 | Theory: why threads break rigid-body contact | the helix SDF, self-locking |
| 3 | **Naïve** threading scene (kinematic coupling) | no self-locking |
| 4 | **SDF-plugin** threading scene | self-locking emerges |
| 5 | Physics-fidelity comparison | self-lock retention, contact force |
| 6 | **Compute-cost** benchmark | ms / `mj_step`, steps/s, SDF overhead |
| 7 | A tiny VLA trained on each regime's demos | success rate on the *faithful* task |
| 8 | Bonus / questions | pitch & gauge sweeps, going further |

**Important note.** Absolute numbers (success rates, exact overhead) are
seed- and hardware-dependent; the *direction* of the comparisons is the lesson.

## 2 · Environment setup & global configuration

### 2.1 Hardware check
A T4 is plenty. Everything also runs on CPU (the physics is small); only the
tiny-policy training benefits from the GPU.

In [ ]:
# ── Hardware verification ──
!nvidia-smi -L || echo "No GPU visible — the notebook will run on CPU (slower but fine)."
import torch
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

### 2.2 Dependencies

The AutoBio thread plugin is shipped as a **precompiled x86-64 Linux shared
object built against MuJoCo 3.3.0** (`libmjlab.so.3.3.0`). It is ABI-locked to
that MuJoCo version, so we pin `mujoco==3.3.0` exactly — a different MuJoCo will
fail to load the plugin.

In [ ]:
# ── Dependency installation (Colab) ──
# mujoco is pinned to 3.3.0 to match the precompiled AutoBio plugin ABI.
!pip install -q "mujoco==3.3.0" imageio "imageio[ffmpeg]" numpy matplotlib
# Headless OpenGL for MuJoCo's renderer on Colab (must be set BEFORE importing mujoco).
import os
os.environ["MUJOCO_GL"] = "egl"
print("MUJOCO_GL =", os.environ["MUJOCO_GL"])

### 2.3 Get the AutoBio plugin + reference scene

We clone the official repo only to obtain two things:
* `autobio/libmjlab.so.3.3.0` — the compiled plugin (registers `mjlab.sdf.thread`,
  `mjlab.sdf.grid`, `mjlab.passive.detent`);
* `autobio/model/scene/screw.xml` — the reference parameters (pitch, radius, gauge)
  for the bolt/nut threads, which we reuse for physical realism.

A fallback message is printed if the clone fails (e.g. offline), so you know
exactly what is missing.

In [ ]:
# ── Clone AutoBio to obtain the compiled plugin and reference scene ──
import os, subprocess, pathlib

REPO_DIR = pathlib.Path("AutoBio")
if not REPO_DIR.exists():
    rc = subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/autobio-bench/AutoBio.git"],
    ).returncode
    if rc != 0:
        print("⚠️  Clone failed. The SDF-plugin sections need 'autobio/libmjlab.so.3.3.0'.")
        print("    The NAÏVE-physics and theory sections still run without it.")

PLUGIN_SO = REPO_DIR / "autobio" / "libmjlab.so.3.3.0"
REF_SCENE = REPO_DIR / "autobio" / "model" / "scene" / "screw.xml"
HAVE_PLUGIN = PLUGIN_SO.exists()
print("plugin .so present:", HAVE_PLUGIN, "->", PLUGIN_SO)

### 2.4 Global imports, seed, plotting & device

In [ ]:
# ── Global configuration ──
import os
os.environ.setdefault("MUJOCO_GL", "egl")

import numpy as np
import mujoco
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.grid": True})
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("mujoco:", mujoco.__version__, "| device:", DEVICE)
assert mujoco.__version__.startswith("3.3"), \
    "The AutoBio plugin requires mujoco 3.3.x; got " + mujoco.__version__

In [ ]:
# ── Load the AutoBio thread plugin (registers mjlab.sdf.thread) ──
# Safe to call once; loading twice raises, so we guard with a flag.
if HAVE_PLUGIN and not globals().get("_PLUGIN_LOADED", False):
    mujoco.mj_loadPluginLibrary(str(PLUGIN_SO))
    _PLUGIN_LOADED = True

# The C-API registry calls (mjp_pluginCount / mjp_getPluginAtSlot) are NOT
# exposed in MuJoCo's Python bindings, so we verify the plugin FUNCTIONALLY:
# compile a tiny model that instantiates mjlab.sdf.thread. If it compiles, the
# plugin is registered and usable.
_PROBE_XML = '''
<mujoco>
  <extension>
    <plugin plugin="mjlab.sdf.thread">
      <instance name="probe">
        <config key="pitch" value="0.003"/>
        <config key="radius" value="0.016"/>
        <config key="low" value="-1.0"/>
        <config key="high" value="0.8"/>
        <config key="gauge" value="0.001"/>
      </instance>
    </plugin>
  </extension>
  <asset><mesh name="probe"><plugin instance="probe"/></mesh></asset>
  <worldbody>
    <geom type="sdf" mesh="probe" contype="0" conaffinity="0">
      <plugin instance="probe"/>
    </geom>
  </worldbody>
</mujoco>'''

PLUGIN_OK = False
if globals().get("_PLUGIN_LOADED", False):
    try:
        mujoco.MjModel.from_xml_string(_PROBE_XML)
        PLUGIN_OK = True
        print("✅ mjlab.sdf.thread loaded and compiles — SDF sections will run.")
    except Exception as e:
        print("⚠️  Plugin .so loaded but the probe model failed to compile:")
        print("   ", repr(e))
        print("    SDF sections will be skipped; naïve + theory sections still run.")
else:
    print("Plugin not loaded — SDF sections will be skipped, naïve sections still run.")
print("PLUGIN_OK =", PLUGIN_OK)

## 3 · Theory — why a screw thread breaks ordinary contact, and how an SDF fixes it

### 3.1 The problem with mesh contact for threads

A bolt/nut pair is two **interlocking helical ramps**. The whole notebook turns
on one question: *how do you represent the contact between those ramps?* There
are three options, and only the third actually works.

| # | Approach | Cheap? | Stable normals? | Real contact force? | Self-locks? |
|---|---|:---:|:---:|:---:|:---:|
| 1 | **Mesh ↔ mesh** — tessellate both threads, collide triangles | ❌ very costly | ❌ jittery | ✅ | ❌ fragile |
| 2 | **Kinematic shortcut** — hinge + slide tied by a constraint | ✅ | — (no contact) | ❌ none | ❌ impossible |
| 3 | **Implicit helix + SDF** — *this plugin* | ✅ | ✅ smooth | ✅ | ✅ emerges |

In one line each:

1. **Mesh ↔ mesh is too expensive and numerically fragile, and can't self-lock.**
   Resolving the contact means colliding two finely tessellated helical meshes:
   *expensive* — many tiny triangles, many candidate contact pairs every step;
   *numerically nasty* — thin, glancing, nearly-tangent contacts give jittery
   normals the solver chokes on; and the delicate friction-cone geometry that
   keeps a tightened cap from spinning back off is too fragile to hold reliably.
2. **The kinematic shortcut is cheap, but it's bookkeeping, not physics.** You
   fake the thread with a hinge + slide tied by an equality constraint. It traces
   a perfect helix going *in*, but there is **no contact surface, no normal
   force, no friction cone** — so nothing resists the reverse motion and
   self-locking is *impossible by construction*.
3. **The right answer is an implicit helix described by a Signed Distance
   Function (SDF).** The thread becomes a *formula*, not a mesh. Φ(P) and its
   gradient (the contact normal) are closed-form, cheap, and smooth — so real
   contact forces, engagement, the transitional bolt–nut state, **and
   self-locking under friction all emerge from the solver** instead of being
   scripted.

The rest of this section makes each claim tangible. First, **three short
interactive demos of why the mesh approach fails** (drag the sliders):

* **3.1·A** — *expensive*: watch the triangle count and candidate contact-pair
  count explode as you refine the mesh;
* **3.1·B** — *numerically nasty*: watch the mesh contact normal jump in jittery
  steps while the true (SDF) normal stays smooth;
* **3.1·C** — *can't self-lock*: watch normal jitter flip a marginally-locked
  thread between “holds” and “backs off”.

Then **3.2** shows the kinematic shortcut and *why releasing it spins the cap
straight back off*, **3.3–3.4** build the SDF that fixes everything, and
**3.4·A** lays the three approaches side by side.


In [ ]:
# ── Interactive-visualisation dependencies (Plotly + ipywidgets) ──
!pip install -q plotly ipywidgets
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from ipywidgets import interact, FloatSlider, Layout

# Colab needs the custom widget manager for live ipywidgets sliders.
try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
    pio.renderers.default = "colab"
except Exception:
    pass

def _S(value, lo, hi, step, desc):
    """A non-jittery slider (updates on release, not on every pixel)."""
    return FloatSlider(value=value, min=lo, max=hi, step=step, description=desc,
                       continuous_update=False, readout_format=".4f",
                       layout=Layout(width="420px"))

import plotly
print("plotly", plotly.__version__, "— drag the sliders below each 3-D plot.")

### 3.1·A Demo — mesh contact is *expensive*

To collide two threads as meshes, each thread is tessellated into a tube of
triangles. Refining the mesh (segments **along** the helix × segments **around**
the tube) multiplies the triangle count, and the brute-force narrow-phase must
consider *every bolt-triangle against every nut-triangle* — an $O(N^2)$ blow-up.
Real engines prune this with a broad-phase, but a thread is the worst case: the
two surfaces are interleaved everywhere, so a huge number of pairs survive
pruning *every single* `mj_step`.

Drag the resolution sliders and watch the counts explode. The SDF does **none**
of this — its cost is a fixed handful of float ops per contact point,
independent of resolution.


In [ ]:
# ── Interactive: mesh tessellation cost (triangles & contact pairs explode) ──
def _tube_mesh(nt, nw, turns, r1=0.16, r2=0.035):
    a  = 0.06/(2*np.pi)                       # visual pitch
    ts = np.linspace(0, turns*2*np.pi, nt)
    ws = np.linspace(0, 2*np.pi, nw, endpoint=False)
    s  = np.sqrt(r1**2 + a**2)
    H  = np.stack([r1*np.cos(ts), r1*np.sin(ts), a*ts], 1)
    T  = np.stack([-r1*np.sin(ts), r1*np.cos(ts), np.full_like(ts, a)], 1)/s
    N  = np.stack([-np.cos(ts), -np.sin(ts), np.zeros_like(ts)], 1)
    B  = np.cross(T, N)
    cw, sw = np.cos(ws)[None, :], np.sin(ws)[None, :]
    X = (H[:, 0:1] + r2*(cw*N[:, 0:1] + sw*B[:, 0:1])).ravel()
    Y = (H[:, 1:2] + r2*(cw*N[:, 1:2] + sw*B[:, 1:2])).ravel()
    Z = (H[:, 2:3] + r2*(cw*N[:, 2:3] + sw*B[:, 2:3])).ravel()
    I, J, Kf = [], [], []
    for i in range(nt-1):
        for j in range(nw):
            a0 = i*nw+j; a1 = i*nw+(j+1) % nw
            b0 = (i+1)*nw+j; b1 = (i+1)*nw+(j+1) % nw
            I += [a0, a0]; J += [a1, b1]; Kf += [b1, b0]
    return X, Y, Z, np.array(I), np.array(J), np.array(Kf)

def mesh_cost_view(along_per_turn=16, around=10, turns=6.0):
    nt = max(2, int(along_per_turn*turns)); nw = int(around)
    X, Y, Z, I, J, Kf = _tube_mesh(nt, nw, turns)
    tris_one  = len(I)
    tris_pair = 2*tris_one                    # bolt + nut
    naive     = tris_one*tris_one             # brute-force narrow-phase pairs
    fig = go.Figure([go.Mesh3d(x=X, y=Y, z=Z, i=I, j=J, k=Kf, color="#b8a93a",
                               flatshading=True, opacity=1.0,
                               lighting=dict(ambient=0.55))])
    fig.update_layout(
        title=f"One thread @ {nt}×{nw} grid  →  {tris_one:,} triangles "
              f"(both parts: {tris_pair:,})",
        height=480, margin=dict(l=0, r=0, t=40, b=0), scene=dict(aspectmode="data"))
    fig.show()
    print(f"triangles / thread        : {tris_one:,}")
    print(f"triangles / bolt+nut pair : {tris_pair:,}")
    print(f"brute-force contact pairs : {naive:,}   (bolt-tris × nut-tris, per mj_step)")
    print(f"SDF equivalent            : O(1) per contact point "
          f"(~a few dozen flops) — independent of resolution")

interact(mesh_cost_view,
         along_per_turn=_S(16, 6, 48, 2, "along / turn"),
         around=_S(10, 6, 32, 2, "around tube"),
         turns=_S(6.0, 1.0, 12.0, 1.0, "# turns"));


### 3.1·B Demo — mesh normals are *numerically nasty*

The solver pushes contacts apart along the **contact normal** and applies
friction perpendicular to it, so a *noisy* normal means noisy forces. A mesh
normal is **piecewise constant** — perpendicular to whichever flat facet the
contact lands on — so it jumps discontinuously as the contact slides across
facet boundaries. For thread flanks, which meet at shallow, nearly-tangent
angles, those jumps are large and frame-to-frame jittery.

Below, a smooth thread flank (black) is approximated by `facets` straight
segments (orange). Slide the contact point: the **mesh normal (orange arrow)**
snaps between facet directions, while the **true / SDF normal (green arrow)**
turns smoothly. The right plot shows the normal *angle* vs. position — a
**staircase** for the mesh against the smooth SDF line. That gap is the jitter
the solver must fight.


In [ ]:
# ── Interactive: faceted (mesh) normal jitter vs smooth (SDF) normal ──
from plotly.subplots import make_subplots

def _flank(x):                       # smooth thread-flank profile (a gentle arc)
    return 0.25*np.sqrt(np.clip(1.0 - (x/1.6)**2, 0, None))

def _normal_angle_from_deriv(d):     # normal = tangent (1,d) rotated +90deg = (-d,1)
    return np.degrees(np.arctan2(1.0, -d))

def mesh_normal_view(facets=8, s=0.55):
    facets = int(facets)
    h  = 1e-4
    xs = np.linspace(-1.5, 1.5, 400); ys = _flank(xs)
    xn = np.linspace(-1.5, 1.5, facets+1); yn = _flank(xn)
    cx = -1.5 + 3.0*s
    # smooth (true) normal
    dt = (_flank(cx+h) - _flank(cx-h))/(2*h)
    n_true = np.array([-dt, 1.0]); n_true /= np.linalg.norm(n_true)
    ang_true = _normal_angle_from_deriv(dt)
    # mesh normal: perpendicular to the facet segment containing cx
    seg = int(np.clip(np.searchsorted(xn, cx)-1, 0, facets-1))
    sx0, sy0, sx1, sy1 = xn[seg], yn[seg], xn[seg+1], yn[seg+1]
    sd = np.array([sx1-sx0, sy1-sy0]); sd /= np.linalg.norm(sd)
    n_mesh = np.array([-sd[1], sd[0]])
    ang_mesh = np.degrees(np.arctan2(n_mesh[1], n_mesh[0]))
    cy = sy0 + (sy1-sy0)*((cx-sx0)/(sx1-sx0))
    # angle-vs-position curves
    grid = np.linspace(-1.5, 1.5, 300); a_true = []; a_mesh = []
    for g in grid:
        a_true.append(_normal_angle_from_deriv((_flank(g+h)-_flank(g-h))/(2*h)))
        k = int(np.clip(np.searchsorted(xn, g)-1, 0, facets-1))
        dd = np.array([xn[k+1]-xn[k], yn[k+1]-yn[k]]); dd /= np.linalg.norm(dd)
        a_mesh.append(np.degrees(np.arctan2(dd[0], -dd[1])))
    fig = make_subplots(rows=1, cols=2, column_widths=[0.55, 0.45],
        subplot_titles=("flank: mesh facets vs smooth surface",
                        "contact-normal angle vs position"))
    L = 0.5
    fig.add_scatter(x=xs, y=ys, mode="lines", line=dict(color="black", width=3),
                    name="true surface", row=1, col=1)
    fig.add_scatter(x=xn, y=yn, mode="lines+markers",
                    line=dict(color="#e08a1e", width=2), name="mesh facets", row=1, col=1)
    fig.add_scatter(x=[cx, cx+L*n_mesh[0]], y=[cy, cy+L*n_mesh[1]], mode="lines",
                    line=dict(color="#e08a1e", width=5), name="mesh normal", row=1, col=1)
    fig.add_scatter(x=[cx, cx+L*n_true[0]], y=[cy, cy+L*n_true[1]], mode="lines",
                    line=dict(color="#2ca02c", width=5), name="SDF normal", row=1, col=1)
    fig.add_scatter(x=[cx], y=[cy], mode="markers",
                    marker=dict(size=10, color="#d9534f"), name="contact", row=1, col=1)
    fig.add_scatter(x=grid, y=a_true, mode="lines",
                    line=dict(color="#2ca02c", width=3), name="SDF (smooth)", row=1, col=2)
    fig.add_scatter(x=grid, y=a_mesh, mode="lines",
                    line=dict(color="#e08a1e", width=2, shape="hv"),
                    name="mesh (staircase)", row=1, col=2)
    fig.add_scatter(x=[cx], y=[ang_true], mode="markers",
                    marker=dict(size=10, color="#d9534f"), showlegend=False, row=1, col=2)
    fig.update_layout(height=440, margin=dict(l=0, r=0, t=50, b=0))
    fig.update_yaxes(scaleanchor="x", row=1, col=1)
    fig.show()
    print(f"normal angle  —  SDF: {ang_true:+6.2f}°   mesh: {ang_mesh:+6.2f}°   "
          f"jitter = {abs(ang_true-ang_mesh):.2f}°   (→ 0 only as facets→∞)")

interact(mesh_normal_view,
         facets=_S(8, 3, 40, 1, "facets"),
         s=_S(0.55, 0.0, 1.0, 0.01, "contact pos"));


### 3.2 The kinematic shortcut — and why it can't self-lock

The obvious way to fake a thread *without* any contact is to give the cap two
joints — a **hinge** about $z$ (turn angle $\theta$) and a **slide** along $z$
(height) — and hard-wire them with an equality constraint so that turning the
cap also lowers it by exactly the pitch:

$$z(\theta) \;=\; -\,\frac{p}{2\pi}\,\theta
\qquad\Longleftrightarrow\qquad
\text{(one full turn } \theta = 2\pi \text{ ⇒ descend by one pitch } p).$$

This is what we build as the **naïve** scene in §4. It *looks* right while you
drive it forward, but it is a purely geometric map with **no contact surface,
no normal force, and no friction cone**. Consequently:

* there is **nothing to resist the reverse motion** — release the cap and
  gravity feeds back through the same rigid coupling, spinning it straight back
  off;
* **self-locking cannot emerge**, because self-locking is a *friction* effect on
  a real contact patch, and here there is no contact at all.

Drag $\theta$ below: the cap rotates **and** descends in lockstep, tracing a
perfect helix — but this is bookkeeping, not physics.

In [ ]:
# ── Interactive: the kinematic hinge↔slide coupling ──
def _cylinder(r, z0, z1, color, opacity, n=40):
    th = np.linspace(0, 2*np.pi, n)
    z = np.array([z0, z1])
    TH, Z = np.meshgrid(th, z)
    X, Y = r*np.cos(TH), r*np.sin(TH)
    return go.Surface(x=X, y=Y, z=Z, showscale=False, opacity=opacity,
                      colorscale=[[0, color], [1, color]], hoverinfo="skip")

def kinematic_view(turns=1.0, pitch=0.03):
    theta = turns * 2*np.pi
    descent = -(pitch/(2*np.pi)) * theta          # = -pitch*turns
    r_shaft, r_cap, cap_h = 0.20, 0.26, 0.18
    cap_z0 = 0.5 + descent
    # Helical trace of a rim marker as the cap was screwed down from 0..theta
    ts = np.linspace(0, theta, 200)
    trace = np.stack([r_cap*np.cos(ts), r_cap*np.sin(ts),
                      0.5 + cap_h - (pitch/(2*np.pi))*ts], 1)
    mk = np.array([r_cap*np.cos(theta), r_cap*np.sin(theta), cap_z0 + cap_h])
    fig = go.Figure([
        _cylinder(r_shaft, 0.0, 0.55, "#b8a93a", 1.0),                 # fixed shaft
        _cylinder(r_cap, cap_z0, cap_z0+cap_h, "#3a7bd5", 0.35),       # moving cap
        go.Scatter3d(x=trace[:,0], y=trace[:,1], z=trace[:,2], mode="lines",
                     line=dict(color="#d9534f", width=6),
                     name="helical path of rim marker"),
        go.Scatter3d(x=[mk[0]], y=[mk[1]], z=[mk[2]], mode="markers",
                     marker=dict(size=6, color="#d9534f"), name="rim marker (θ)"),
        go.Scatter3d(x=[0,mk[0]], y=[0,mk[1]], z=[mk[2],mk[2]], mode="lines",
                     line=dict(color="#222", width=4), name="orientation"),
    ])
    fig.update_layout(
        title=f"Kinematic coupling:  θ = {turns:.2f} turns  →  descent = {descent:+.3f}"
              f"   (NO contact, NO friction ⇒ cannot self-lock)",
        height=520, margin=dict(l=0,r=0,t=40,b=0),
        scene=dict(aspectmode="data", xaxis_title="x", yaxis_title="y", zaxis_title="z"))
    fig.show()

interact(kinematic_view,
         turns=_S(1.0, 0.0, 3.0, 0.05, "θ (turns)"),
         pitch=_S(0.03, 0.005, 0.08, 0.005, "pitch p"));

### 3.3 The thread as a parametric helix tube $(r_1, r_2, p, l, h)$

The plugin replaces the mesh with an **implicit helix**. The thread *centreline*
is a circular helix of radius $r_1$ and pitch $p$ (axial rise per full turn):

$$\mathbf{H}(t)\;=\;\Big[\,r_1\cos t,\;\; r_1\sin t,\;\; \tfrac{p}{2\pi}\,t\,\Big]^{\!\top},
\qquad t\in[\,l,\,h\,].$$

The solid thread is the **tube of radius $r_2$** swept along that centreline
(so $r_2$ is the thread half-thickness, the `gauge`). A full thread is therefore
described by the tuple

$$\boxed{(\,r_1,\; r_2,\; p,\; l,\; h\,)}$$

| symbol | meaning | screw.xml analogue |
|---|---|---|
| $r_1$ | helix (centreline) radius | `radius` |
| $r_2$ | tube radius / thread thickness | `gauge` |
| $p$ | pitch — axial rise per full turn | `pitch` |
| $l, h$ | parameter bounds; #turns $=(h-l)/2\pi$ | `low`, `high` |

We build the tube surface from the helix's Frenet frame:
$\mathbf{T}=\mathbf{H}'/\lVert\mathbf{H}'\rVert$, principal normal
$\mathbf{N}=[-\cos t,-\sin t,0]$, binormal $\mathbf{B}=\mathbf{T}\times\mathbf{N}$,
and sweep a circle of radius $r_2$:
$\;\mathbf{S}(t,\omega)=\mathbf{H}(t)+r_2(\cos\omega\,\mathbf{N}+\sin\omega\,\mathbf{B})$.

Drag the **five** sliders and watch the thread reshape.

In [ ]:
# ── Helix geometry + SDF helpers (reused by both 3-D plots below) ──
def helix_point(t, r1, p):
    """Single point H(t) on the helix centreline (pitch p = rise per full turn)."""
    a = p/(2*np.pi)
    return np.array([r1*np.cos(t), r1*np.sin(t), a*t])

def build_tube(r1, r2, p, l, h, nt=180, nw=26):
    """Tube of radius r2 swept along the helix centreline via its Frenet frame."""
    a = p/(2*np.pi)
    ts = np.linspace(l, h, nt)
    ws = np.linspace(0, 2*np.pi, nw)
    H = np.stack([r1*np.cos(ts), r1*np.sin(ts), a*ts], 1)             # (nt,3)
    s = np.sqrt(r1**2 + a**2)
    T = np.stack([-r1*np.sin(ts), r1*np.cos(ts), np.full_like(ts, a)], 1) / s
    N = np.stack([-np.cos(ts), -np.sin(ts), np.zeros_like(ts)], 1)
    B = np.cross(T, N)
    cw, sw = np.cos(ws)[None, :], np.sin(ws)[None, :]                 # (1,nw)
    X = H[:,0:1] + r2*(cw*N[:,0:1] + sw*B[:,0:1])
    Y = H[:,1:2] + r2*(cw*N[:,1:2] + sw*B[:,1:2])
    Z = H[:,2:3] + r2*(cw*N[:,2:3] + sw*B[:,2:3])
    return X, Y, Z, H

def _newton_t(t, P, r1, a, iters=8):
    """Refine the helix parameter t to minimise ||P - H(t)||^2 (Newton's method)."""
    for _ in range(iters):
        Ht  = np.array([r1*np.cos(t),  r1*np.sin(t),  a*t])
        Hp  = np.array([-r1*np.sin(t), r1*np.cos(t),  a])      # H'(t)
        Hpp = np.array([-r1*np.cos(t), -r1*np.sin(t), 0.0])    # H''(t)
        d = Ht - P
        grad = 2.0 * d @ Hp
        hess = 2.0 * (Hp @ Hp + d @ Hpp)
        if abs(hess) < 1e-12:
            break
        t = t - grad/hess
    return t

def helix_sdf(P, r1, r2, p, l, h):
    """Signed distance from point P to the helix tube of radius r2.

    Step 1: wrap angle phi = atan2(P.y, P.x).
    Step 2: winding k = round(P.z/p - phi/2pi) gives the analytic SEED t = phi+2pi*k.
            Because the true closest point can sit on the neighbouring turn, we
            seed k-1, k, k+1, Newton-refine each, and keep the nearest — so the
            returned t* and distance are essentially exact (~1e-8 m).
    Step 3: Phi(P) = ||P - H(t*)|| - r2   (negative = inside the solid thread).
    Returns the winding k, parameter t*, nearest centreline point, and Phi.
    """
    px, py, pz = P
    phi = np.arctan2(py, px)                          # wrap angle in (-pi, pi]
    a = p/(2*np.pi)
    k0 = int(np.round(pz/p - phi/(2*np.pi)))          # analytic winding seed
    best = None
    for k in (k0 - 1, k0, k0 + 1):
        t = float(np.clip(_newton_t(phi + 2*np.pi*k, P, r1, a), l, h))
        d = float(np.linalg.norm(P - helix_point(t, r1, p)))
        if best is None or d < best[0]:
            best = (d, t, k)
    dist, tstar, k = best
    return dict(phi=float(phi), k=k, tstar=tstar,
                near=helix_point(tstar, r1, p), sdf=dist - r2)

print("helpers ready: helix_point, build_tube, helix_sdf")

In [ ]:
# ── Interactive: reshape the helix tube with the 5 parameters (r1,r2,p,l,h) ──
def helix_view(r1=0.016, r2=0.002, p=0.006, l=-9.4, h=9.4):
    X, Y, Z, H = build_tube(r1, r2, p, l, h)
    fig = go.Figure([
        go.Surface(x=X, y=Y, z=Z, colorscale="Viridis", showscale=False,
                   opacity=0.95, name="thread tube"),
        go.Scatter3d(x=H[:,0], y=H[:,1], z=H[:,2], mode="lines",
                     line=dict(color="black", width=4), name="centreline H(t)"),
    ])
    turns = (h - l)/(2*np.pi)
    fig.update_layout(
        title=f"Helix tube  (r1={r1:.3f}, r2={r2:.4f}, p={p:.3f}, "
              f"l={l:.2f}, h={h:.2f})  →  {turns:.2f} turns",
        height=560, margin=dict(l=0,r=0,t=40,b=0),
        scene=dict(aspectmode="data"))
    fig.show()

interact(helix_view,
         r1=_S(0.016, 0.005, 0.03,  0.001, "r1 radius"),
         r2=_S(0.002, 0.0005,0.006, 0.0005,"r2 gauge"),
         p =_S(0.006, 0.002, 0.02,  0.001, "p pitch"),
         l =_S(-9.4, -18.8,  0.0,   0.4,   "l low"),
         h =_S( 9.4,  0.0,   18.8,  0.4,   "h high"));

### 3.4 The signed distance function: solving for $t^\star$ and $k$

Given a query point $\mathbf{P}=(p_x,p_y,p_z)$, the SDF returns the distance to
the tube surface. The only hard part is finding the **closest centreline point**
$\mathbf{H}(t^\star)$. The trick that makes it *analytic* (no search):

**Step 1 — wrap angle.** $\;\varphi=\operatorname{atan2}(p_y,p_x)$. The helix
passes through this angle at every $t=\varphi+2\pi k,\ k\in\mathbb{Z}$ — once per
turn.

**Step 2 — pick the turn $k$.** Among those candidates, choose the one whose
*height* $\tfrac{p}{2\pi}t$ is closest to $p_z$:

$$k \;=\; \operatorname{round}\!\Big(\frac{p_z}{p} - \frac{\varphi}{2\pi}\Big),
\qquad t^\star \;=\; \operatorname{clip}\big(\varphi + 2\pi k,\; l,\; h\big).$$

So **$k$ is the winding number** — *which turn of the thread* the nearest point
lies on — and $\varphi+2\pi k$ is the **analytic seed** for the helix parameter.
The closed form gets the right *turn*; a couple of **Newton steps** on
$\lVert\mathbf{P}-\mathbf{H}(t)\rVert^2$ then slide $t$ to the exact nearest
$t^\star$ (the code seeds $k\!-\!1,k,k\!+\!1$ and keeps the closest, so the
picture is accurate to $\sim\!10^{-8}\,$m). This analytic-seed-then-refine is
exactly why the SDF is cheap: no global search, no mesh.

**Step 3 — signed distance.**

$$\Phi(\mathbf{P}) \;=\; \big\lVert\,\mathbf{P}-\mathbf{H}(t^\star)\,\big\rVert \;-\; r_2,
\qquad \Phi<0 \text{ inside the solid thread.}$$

Both $\Phi$ and its gradient $\nabla\Phi=\dfrac{\mathbf{P}-\mathbf{H}(t^\star)}
{\lVert\mathbf{P}-\mathbf{H}(t^\star)\rVert}$ (the contact normal) are closed-form —
no mesh, no broad-phase. That cheapness, plus a smooth signed field, is what lets
engagement, the transitional bolt–nut state, **and self-locking under friction**
all emerge from the solver instead of being scripted.

Drag the point $(p_x,p_y,p_z)$ around the thread and watch $k$, $t^\star$ and
$\Phi$ update; the dashed line is $\mathbf{P}\!\to\!\mathbf{H}(t^\star)$.

In [ ]:
# ── Interactive: pick a point P, compute the SDF (shows t*, k, Phi) ──
def sdf_view(r1=0.016, r2=0.002, p=0.006,
             px=0.024, py=0.0, pz=0.004, l=-12.6, h=12.6):
    P = np.array([px, py, pz])
    X, Y, Z, H = build_tube(r1, r2, p, l, h)
    r = helix_sdf(P, r1, r2, p, l, h)
    inside = r["sdf"] < 0
    pcol = "#d9534f" if inside else "#1f77b4"
    fig = go.Figure([
        go.Surface(x=X, y=Y, z=Z, colorscale="Greys", showscale=False,
                   opacity=0.55, name="thread tube"),
        go.Scatter3d(x=H[:,0], y=H[:,1], z=H[:,2], mode="lines",
                     line=dict(color="black", width=3), name="centreline"),
        go.Scatter3d(x=[px], y=[py], z=[pz], mode="markers",
                     marker=dict(size=7, color=pcol, symbol="diamond"),
                     name="query point P"),
        go.Scatter3d(x=[r["near"][0]], y=[r["near"][1]], z=[r["near"][2]],
                     mode="markers", marker=dict(size=6, color="#2ca02c"),
                     name="nearest H(t*)"),
        go.Scatter3d(x=[px, r["near"][0]], y=[py, r["near"][1]],
                     z=[pz, r["near"][2]], mode="lines",
                     line=dict(color="#2ca02c", width=5, dash="dash"),
                     name="distance"),
    ])
    state = ("INSIDE solid" if inside else "outside")
    fig.update_layout(
        title=(f"k = {r['k']}   t* = {r['tstar']:.3f} rad   "
               f"Φ(P) = {r['sdf']*1000:+.3f} mm   ({state})"),
        height=560, margin=dict(l=0,r=0,t=40,b=0), scene=dict(aspectmode="data"))
    fig.show()
    print(f"φ(P)={r['phi']:.3f} rad → winding k={r['k']} (which turn), "
          f"t*={r['tstar']:.3f} rad, |P-H(t*)|={np.linalg.norm(P-r['near'])*1000:.3f} mm, "
          f"gauge r2={r2*1000:.3f} mm  ⇒  Φ={r['sdf']*1000:+.3f} mm")

interact(sdf_view,
         r1=_S(0.016, 0.005, 0.03,  0.001, "r1 radius"),
         r2=_S(0.002, 0.0005,0.006, 0.0005,"r2 gauge"),
         p =_S(0.006, 0.002, 0.02,  0.001, "p pitch"),
         px=_S(0.024,-0.04, 0.04,  0.002, "P.x"),
         py=_S(0.0,  -0.04, 0.04,  0.002, "P.y"),
         pz=_S(0.004,-0.03, 0.03,  0.002, "P.z"),
         l =_S(-12.6,-18.8, 0.0,   0.4,   "l low"),
         h =_S( 12.6, 0.0,  18.8,  0.4,   "h high"));

### 3.5 The real parameters AutoBio uses

In MJCF the helix SDF is attached as a `<geom type="sdf">` backed by a
`<plugin instance="…">`. Here are the exact `mjlab.sdf.thread` parameters from
the repo's `screw.xml` — the same $(r_1\!\approx\!\texttt{radius},\,
r_2\!\approx\!\texttt{gauge},\,p\!=\!\texttt{pitch},\,l\!=\!\texttt{low},\,
h\!=\!\texttt{high})$ tuple you just explored:

In [ ]:
# ── Show the reference thread parameters from AutoBio's screw.xml ──
if REF_SCENE.exists():
    txt = REF_SCENE.read_text()
    start = txt.find("<plugin plugin=\"mjlab.sdf.thread\">")
    end = txt.find("</plugin>", start) + len("</plugin>")
    print(txt[start:end] if start != -1 else "(plugin block not found)")
else:
    print("screw.xml not available; using the documented defaults below.")

# Canonical thread parameters (from screw.xml). We reuse these for realism.
THREAD = dict(pitch=0.003, bolt_radius=0.016, nut_radius=0.017,
              low=-1.0, high=0.8, bolt_gauge=0.001, nut_gauge=0.0009)
print("THREAD parameters:", THREAD)

## 4 · Two threading scenes that differ *only* in their physics

To isolate the effect of the plugin we strip away the robot arm and lab clutter
of the full AutoBio task and keep a **minimal cap-on-tube screwing scene**. The
cap has the same two joints in both versions:

* a **hinge** about $z$ (the screwing rotation), and
* a **slide** along $z$ (the cap's vertical travel).

The *only* difference between the two scenes is how rotation couples to
descent and whether real thread contact exists:

| | **Naïve** (no plugin) | **SDF** (AutoBio plugin) |
|---|---|---|
| rotation→descent | hard `equality` coupling (kinematic) | emerges from helix-SDF **contact** |
| thread contact | none | `mjlab.sdf.thread` bolt vs nut |
| self-locking | impossible (kinematic) | emerges from contact friction |

We position a fixed camera so we can later render observations for the policy.

In [ ]:
# ── Shared scene fragments ──
PITCH = THREAD["pitch"]                 # metres of descent per full turn (2*pi rad)
SLIDE_PER_RAD = -PITCH / (2 * np.pi)    # descend (negative z) as hinge angle grows

CAMERA = '''
    <camera name="obs" pos="0.12 -0.12 0.16" xyaxes="0.7 0.7 0 -0.25 0.25 0.94"/>
'''

LIGHTS_FLOOR = '''
    <light directional="true" diffuse="0.7 0.7 0.7" ambient="0.35 0.35 0.35"
           pos="0 0 1" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="1 1 0.05" rgba="0.8 0.85 0.9 1"/>
'''

# ── ENGAGEMENT FIXES (so the threads actually mesh, and nothing runs away) ──
# Original bug: cap_top/cap_wall were SOLID collidable cylinders that bottomed
# on the shaft top after ~4 mm, and the nut SDF sat ~28 mm above the bolt SDF, so
# the helices never met. Fixes shared by BOTH scenes (they still differ ONLY in
# physics):
#   1. cap shell is NON-COLLIDING (contype=0 conaffinity=0) and the cap is
#      lowered to surround the post top (so, in the SDF scene, the nut SDF can be
#      co-located with the bolt SDF and the helices interlock).
#   2. joint DAMPING + a slide RANGE limit so the kinematic cap can't run away
#      (with the seating gone, the undamped naive coupling was numerically
#      unstable).

def naive_scene_xml():
    """Naïve threading: hinge and slide hard-coupled by an equality constraint.

    Turning the cap lowers it at exactly the thread pitch, but nothing resists
    the reverse motion: there is no contact and hence no self-locking.
    """
    return f'''
<mujoco model="naive_thread">
  <option timestep="0.001" integrator="implicitfast"/>
  <worldbody>
    {LIGHTS_FLOOR}{CAMERA}
    <!-- fixed tube + threaded shaft -->
    <body name="tube" pos="0 0 0.05">
      <geom name="tube" type="cylinder" size="0.016 0.05" rgba="0.5 0.5 0.55 1"/>
      <geom name="shaft" type="cylinder" size="0.016 0.012" pos="0 0 0.062" rgba="0.6 0.6 0.2 1"/>
    </body>
    <!-- cap (shell non-colliding) on a hinge(z) + slide(z), surrounding the post top -->
    <body name="cap" pos="0 0 0.12">
      <joint name="hinge" type="hinge" axis="0 0 1" damping="0.02"/>
      <joint name="slide" type="slide" axis="0 0 1" damping="0.02"
             range="-0.013 0.012" limited="true"/>
      <geom name="cap_top"  type="cylinder" size="0.022 0.003" pos="0 0 0.033"
            rgba="0.2 0.5 0.9 1"    contype="0" conaffinity="0"/>
      <geom name="cap_wall" type="cylinder" size="0.022 0.020" pos="0 0 0.010"
            rgba="0.2 0.5 0.9 0.35" contype="0" conaffinity="0"/>
    </body>
  </worldbody>
  <equality>
    <!-- slide = SLIDE_PER_RAD * hinge  (pure kinematic thread, no contact) -->
    <joint joint1="slide" joint2="hinge" polycoef="0 {SLIDE_PER_RAD} 0 0 0"/>
  </equality>
  <actuator>
    <position name="screw" joint="hinge" kp="3.0"/>
  </actuator>
</mujoco>
'''

print(naive_scene_xml()[:400], "...")


In [ ]:
# ── SDF (faithful) scene: identical joints, but threading comes from contact ──
def sdf_scene_xml():
    """Faithful threading using AutoBio's helix-SDF plugin.

    The cap keeps the same hinge+slide joints and the same (non-colliding) shell
    as the naïve scene; the ONLY difference is that there is no equality coupling
    and a bolt-thread SDF (on the shaft) meets a nut-thread SDF (in the cap). The
    nut is co-located with the bolt and the helices span ~4.5 turns (low/high =
    +/-14 rad) so they stay meshed through the whole descent. The nut radius has
    a little clearance over the bolt (0.0177 vs 0.016 + gauges) so the threads
    mesh with LIGHT contact instead of jamming; rotation->descent and
    self-locking emerge from that contact and its friction.
    """
    t = THREAD
    return f'''
<mujoco model="sdf_thread">
  <extension>
    <plugin plugin="mjlab.sdf.thread">
      <instance name="bolt">
        <config key="pitch" value="{t['pitch']}"/>
        <config key="radius" value="{t['bolt_radius']}"/>
        <config key="low" value="-14.0"/>
        <config key="high" value="14.0"/>
        <config key="gauge" value="{t['bolt_gauge']}"/>
      </instance>
      <instance name="nut">
        <config key="pitch" value="{t['pitch']}"/>
        <config key="radius" value="0.0177"/>
        <config key="low" value="-14.0"/>
        <config key="high" value="14.0"/>
        <config key="gauge" value="{t['nut_gauge']}"/>
      </instance>
    </plugin>
  </extension>

  <option timestep="0.001" integrator="implicitfast" impratio="1" cone="elliptic"
          noslip_iterations="2">
    <flag multiccd="enable"/>
  </option>

  <asset>
    <mesh name="bolt"><plugin instance="bolt"/></mesh>
    <mesh name="nut"><plugin instance="nut"/></mesh>
  </asset>

  <worldbody>
    {LIGHTS_FLOOR}{CAMERA}
    <body name="tube" pos="0 0 0.05">
      <geom name="tube" type="cylinder" size="0.016 0.05" rgba="0.5 0.5 0.55 1"/>
      <geom name="shaft" type="cylinder" size="0.016 0.012" pos="0 0 0.062" rgba="0.6 0.6 0.2 1"/>
      <geom name="bolt" type="sdf" mesh="bolt" pos="0 0 0.062" rgba="0.6 0.6 0.2 1"
            contype="0" conaffinity="0"><plugin instance="bolt"/></geom>
    </body>
    <!-- cap lowered to 0.12 so its nut SDF (pos -0.004 -> world z 0.116) sits
         just above the bolt SDF (world z 0.112): the helices interlock. Shell
         geoms are non-colliding so only the nut<->bolt SDF pair governs contact. -->
    <body name="cap" pos="0 0 0.12">
      <joint name="hinge" type="hinge" axis="0 0 1" damping="0.02"/>
      <joint name="slide" type="slide" axis="0 0 1" damping="0.02"
             range="-0.013 0.012" limited="true"/>
      <geom name="cap_top"  type="cylinder" size="0.022 0.003" pos="0 0 0.033"
            rgba="0.2 0.5 0.9 1"    contype="0" conaffinity="0"/>
      <geom name="cap_wall" type="cylinder" size="0.022 0.020" pos="0 0 0.010"
            rgba="0.2 0.5 0.9 0.35" contype="0" conaffinity="0"/>
      <geom name="nut" type="sdf" mesh="nut" pos="0 0 -0.004" rgba="0.9 0.3 0.3 1"
            contype="0" conaffinity="0"><plugin instance="nut"/></geom>
    </body>
  </worldbody>

  <contact>
    <pair geom1="nut" geom2="bolt" friction="0.3 0.3 0.005 0.0001 0.0001"
          solref="0.01 1"/>
  </contact>

  <actuator>
    <position name="screw" joint="hinge" kp="3.0"/>
  </actuator>
</mujoco>
'''

print("SDF scene XML ready (will only load if PLUGIN_OK).")


> **Note on stiff thread contact.** SDF thread contact is stiff; AutoBio uses
> `impratio=1`, an elliptic friction cone, `multiccd`, and noslip iterations
> (copied above). If you see jitter, reduce `timestep` to `0.001` or soften
> `solref`. These are exactly the knobs the paper tunes.

### 4.1 Helpers — build, render, and a screwing controller

In [ ]:
# ── Model helpers ──
def make_model(xml):
    return mujoco.MjModel.from_xml_string(xml)

def jid(model, name):
    return model.joint(name).qposadr.item()

# IMPORTANT: a mujoco.Renderer allocates a GL/EGL context. Creating one PER FRAME
# (e.g. inside a rollout loop) leaks contexts and will hang/crash the Colab
# runtime after a few thousand frames. We cache one renderer per (model, size)
# and REUSE it. Build the model ONCE per scene so the cache key stays stable.
_RENDERER_CACHE = {}

def get_renderer(model, width, height):
    """Return a cached Renderer for this (model, size); create it only once."""
    key = (id(model), width, height)
    r = _RENDERER_CACHE.get(key)
    if r is None:
        r = mujoco.Renderer(model, height, width)
        _RENDERER_CACHE[key] = r
    return r

def render_rgb(model, data, width=96, height=96, cam="obs"):
    """Render a single RGB frame from the 'obs' camera (headless EGL), reusing
    a cached renderer so we never leak GL contexts inside loops."""
    r = get_renderer(model, width, height)
    r.update_scene(data, camera=cam)
    return r.render().copy()

def screw_controller(model, data, target_angle, hold_steps=400, kp_track=True):
    """Drive the hinge toward target_angle, logging trajectory.

    Returns a dict of per-step logs: hinge, slide (descent), and (if present)
    the max thread contact force magnitude.
    """
    h_adr, s_adr = jid(model, "hinge"), jid(model, "slide")
    log = {"hinge": [], "slide": [], "cforce": [], "frames_t": []}
    n = int(target_angle / 0.02) + hold_steps          # ~0.02 rad per step ramp
    for k in range(n):
        ramp = min(1.0, k / max(1, (n - hold_steps)))
        data.ctrl[0] = target_angle * ramp
        mujoco.mj_step(model, data)
        log["hinge"].append(data.qpos[h_adr])
        log["slide"].append(data.qpos[s_adr])
        # contact force magnitude (0 in the naïve scene, which has no contact)
        f = 0.0
        for c in range(data.ncon):
            tmp = np.zeros(6); mujoco.mj_contactForce(model, data, c, tmp)
            f = max(f, float(np.linalg.norm(tmp[:3])))
        log["cforce"].append(f)
    return {k: np.array(v) for k, v in log.items() if k != "frames_t"}

### 4.2 Diagnostic — do the two helices actually engage?

The fixes above are reasoned, not yet verified (the SDF plugin only runs on
Colab/Linux). This cell is an instrument: it prints the bolt and nut geom
positions and bounding radii, then screws the cap a little and reports, every
100 steps, **how many contacts exist, which geoms are touching, the peak contact
force, and the descent**.

**What success looks like:** a `('nut','bolt')` pair appears in `pairs` with a
non-zero `fmax`, and `descent` grows smoothly as the cap turns. If instead
`ncon=0` / no nut-bolt pair, the helices still aren't meeting — paste the output
back and we'll adjust the nut `pos`, the thread `low/high`, or the radii.


In [ ]:
# ── Diagnostic: does the nut SDF actually engage the bolt SDF? ──
if PLUGIN_OK:
    md_ = make_model(sdf_scene_xml()); dd_ = mujoco.MjData(md_)
    gname = lambda i: mujoco.mj_id2name(md_, mujoco.mjtObj.mjOBJ_GEOM, i)
    bolt_id, nut_id = md_.geom("bolt").id, md_.geom("nut").id
    mujoco.mj_forward(md_, dd_)
    print(f"bolt geom: world pos {dd_.geom_xpos[bolt_id].round(4)}  rbound {md_.geom_rbound[bolt_id]:.4f} m")
    print(f"nut  geom: world pos {dd_.geom_xpos[nut_id].round(4)}  rbound {md_.geom_rbound[nut_id]:.4f} m")
    print("(rbound is the geom's bounding radius - tells us how far the thread extends)\n")

    s_adr = jid(md_, "slide")
    for k in range(600):
        dd_.ctrl[0] = 0.02 * k
        mujoco.mj_step(md_, dd_)
        if k % 100 == 0:
            pairs, fmax = [], 0.0
            for c in range(dd_.ncon):
                con = dd_.contact[c]
                pairs.append((gname(con.geom1), gname(con.geom2)))
                tmp = np.zeros(6); mujoco.mj_contactForce(md_, dd_, c, tmp)
                fmax = max(fmax, float(np.linalg.norm(tmp[:3])))
            nutbolt = any(set(p) == {"nut", "bolt"} for p in pairs)
            print(f"step {k:4d}: ncon={dd_.ncon}  nut<->bolt={nutbolt}  "
                  f"fmax={fmax:6.3f} N  descent={-dd_.qpos[s_adr]*1000:6.2f} mm  pairs={pairs}")
    print("\nSUCCESS = a nut<->bolt pair with fmax>0 and descent growing. "
          "If not, paste this output back.")
else:
    print("Plugin unavailable - cannot run the SDF engagement diagnostic.")


## 5 · Physics-fidelity comparison

We screw the cap down by the same commanded angle in both scenes, then run a
**true self-locking test**.

The earlier versions of this test were misleading. Holding the hinge with the
position servo pinned *both* scenes (nothing could move). Pulling the cap off
with an **axial force** is also wrong: the rigid kinematic `equality` is a
1-DOF helix constraint that simply *reacts* an axial load (it is geometrically
self-locking against pull-off), while a large axial force just rips the
contact-only SDF cap straight off — the opposite of what we want to show.

The load that actually discriminates **friction from no-friction** is an
**unscrewing torque** on the hinge. So the corrected test:

1. **Releases the actuator for real** — its gain is zeroed, so it exerts no torque.
2. **Applies a slowly *ramped* loosening torque** (0 → `LOOSEN_TORQUE`) to the
   hinge, plus light joint damping so nothing runs away.

Expected, and robust to the exact friction value:

* the **naïve** coupling has *zero* friction, so it back-drives from the very
  first instant — its curve peels away from step 0;
* the **SDF** thread holds until the torque exceeds its static-friction
  **breakaway**, so its curve stays flat for a while first. **That flat delay is
  the self-locking margin.**


In [ ]:
# ── Screw down + a TRUE self-locking test (ramped loosening torque) ──
TARGET_ANGLE    = 6 * np.pi      # three full turns
LOOSEN_TORQUE   = 0.03           # N·m peak unscrewing torque, ramped 0→this on release
RELEASE_DAMPING = 0.01           # joint damping during release (keeps motion bounded)

def screw_then_release(xml, release_steps=1500):
    model = make_model(xml)
    data  = mujoco.MjData(model)
    log   = screw_controller(model, data, TARGET_ANGLE, hold_steps=300)
    locked_start = data.qpos[jid(model, "slide")]
    h_adr, s_adr = jid(model, "hinge"), jid(model, "slide")
    h_dof = model.joint("hinge").dofadr.item()

    # --- TRUE release: kill the actuator, then apply a slowly ramped unscrewing
    #     torque. Naïve (no friction) back-drives at once; the SDF thread holds
    #     until the torque exceeds its static-friction breakaway. ---
    model.actuator_gainprm[0, 0] = 0.0      # position servo now exerts zero torque
    model.actuator_biasprm[0, 1] = 0.0
    model.dof_damping[:] = RELEASE_DAMPING
    data.ctrl[0] = 0.0

    rel = {"hinge": [], "slide": []}
    for k in range(release_steps):
        data.qfrc_applied[h_dof] = -LOOSEN_TORQUE * (k / release_steps)   # unscrew
        mujoco.mj_step(model, data)
        rel["hinge"].append(data.qpos[h_adr]); rel["slide"].append(data.qpos[s_adr])
    return model, data, log, {k: np.array(v) for k, v in rel.items()}, locked_start

def _turns(h):
    return (h[-1] - h[0]) / (2 * np.pi)

m_n, d_n, log_n, rel_n, lock_n = screw_then_release(naive_scene_xml())
print(f"NAÏVE  | descent at tighten: {-lock_n*1000:.2f} mm | "
      f"unwound after release: {_turns(rel_n['hinge']):+.3f} turns")

if PLUGIN_OK:
    m_s, d_s, log_s, rel_s, lock_s = screw_then_release(sdf_scene_xml())
    print(f"SDF    | descent at tighten: {-lock_s*1000:.2f} mm | "
          f"unwound after release: {_turns(rel_s['hinge']):+.3f} turns")
    print("Expect: naïve unwinds steadily from step 0; SDF holds (≈0) until breakaway.")
else:
    print("SDF scene skipped (plugin unavailable).")


In [ ]:
# ── Plot: descent while screwing, and how far each unwinds on release ──
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(-log_n["slide"]*1000, label="naïve")
if PLUGIN_OK: ax[0].plot(-log_s["slide"]*1000, label="SDF plugin")
ax[0].set(title="Descent while screwing", xlabel="step", ylabel="cap descent (mm)")
ax[0].legend()

ax[1].plot((rel_n["hinge"]-rel_n["hinge"][0])/(2*np.pi), label="naïve")
if PLUGIN_OK:
    ax[1].plot((rel_s["hinge"]-rel_s["hinge"][0])/(2*np.pi), label="SDF plugin")
ax[1].axhline(0, color="gray", ls=":", lw=1)
ax[1].set(title="Self-locking test (released + ramped loosening torque)",
          xlabel="step after release", ylabel="cap unwound (turns)")
ax[1].legend()
plt.tight_layout(); plt.show()
print("naïve peels away from step 0 (no friction to resist); SDF stays flat until "
      "the ramped torque passes its self-locking breakaway.")


In [ ]:
# ── Visual sanity check: render the final tightened state ──
imgs = [("naïve", render_rgb(m_n, d_n))]
if PLUGIN_OK: imgs.append(("SDF plugin", render_rgb(m_s, d_s)))
fig, ax = plt.subplots(1, len(imgs), figsize=(4*len(imgs), 4))
ax = np.atleast_1d(ax)
for a, (name, im) in zip(ax, imgs):
    a.imshow(im); a.set_title(name); a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ── Contact force: only the SDF scene develops real thread contact ──
if PLUGIN_OK:
    plt.figure(figsize=(7,3.5))
    plt.plot(log_s["cforce"], label="SDF thread contact force")
    plt.plot(log_n["cforce"], label="naïve (no contact)")
    plt.title("Max thread contact-force magnitude while tightening")
    plt.xlabel("step"); plt.ylabel("force (N)"); plt.legend(); plt.show()
    print("The naïve scene reports ~0 N: its 'thread' is a kinematic constraint, "
          "not a physical contact.")
else:
    print("Plugin unavailable — contact-force comparison needs the SDF scene.")

### 5.1 An animated video of the whole tightening + release

Numbers and static frames only go so far. Here we render the **entire event** —
screw down, then the true release with the pop-off load — as a side-by-side
video: **left = naïve, right = SDF**. A green banner marks the *screwing* phase,
red marks *released*. Watch the SDF cap hold while the naïve cap unscrews and
lifts off.

> **Tuning.** If the **SDF** cap also drifts, lower `BACKOFF_FORCE`; if the
> **naïve** cap barely moves, raise `BACKOFF_FORCE` or lower `RELEASE_DAMPING`
> (both defined in §5). Rendering a few hundred frames takes ~10–30 s.


In [ ]:
# ── Roll out screw-down + true release, capturing frames for a video ──
def rollout_frames(xml, release_steps=1500, frame_every=5, W=300, H=340):
    model = make_model(xml); data = mujoco.MjData(model)
    h_adr, s_adr = jid(model, "hinge"), jid(model, "slide")
    h_dof = model.joint("hinge").dofadr.item()
    frames, fphase, slide = [], [], []
    n = int(TARGET_ANGLE / 0.02) + 300
    for k in range(n):                                   # ---- screwing ----
        ramp = min(1.0, k / max(1, (n - 300)))
        data.ctrl[0] = TARGET_ANGLE * ramp
        mujoco.mj_step(model, data)
        slide.append(data.qpos[s_adr])
        if k % frame_every == 0:
            frames.append(render_rgb(model, data, W, H)); fphase.append(0)
    model.actuator_gainprm[0, 0] = 0.0                   # ---- true release ----
    model.actuator_biasprm[0, 1] = 0.0
    model.dof_damping[:] = RELEASE_DAMPING
    data.ctrl[0] = 0.0
    for k in range(release_steps):
        data.qfrc_applied[h_dof] = -LOOSEN_TORQUE * (k / release_steps)
        mujoco.mj_step(model, data)
        slide.append(data.qpos[s_adr])
        if k % frame_every == 0:
            frames.append(render_rgb(model, data, W, H)); fphase.append(1)
    return dict(frames=frames, fphase=fphase, slide=np.array(slide))

print("Rendering naïve rollout…"); vid_n = rollout_frames(naive_scene_xml())
if PLUGIN_OK:
    print("Rendering SDF rollout…"); vid_s = rollout_frames(sdf_scene_xml())
print("frames:", len(vid_n["frames"]), "(naïve) /",
      len(vid_s["frames"]) if PLUGIN_OK else "-", "(SDF)")


In [ ]:
# ── Assemble the side-by-side video and display it ──
import imageio
from IPython.display import Video

def _banner(img, label, phase_val):
    img = img.copy()
    img[:10, :, :] = (60, 160, 60) if phase_val == 0 else (200, 60, 60)
    try:
        from PIL import Image, ImageDraw
        im = Image.fromarray(img); dr = ImageDraw.Draw(im)
        dr.text((5, 12), f"{label} — {'screwing' if phase_val == 0 else 'RELEASED'}",
                fill=(255, 255, 255))
        img = np.array(im)
    except Exception:
        pass
    return img

frames_n, ph_n = vid_n["frames"], vid_n["fphase"]
m = len(frames_n) if not PLUGIN_OK else min(len(frames_n), len(vid_s["frames"]))
clip = []
for i in range(m):
    left = _banner(frames_n[i], "naïve", ph_n[i])
    if PLUGIN_OK:
        right = _banner(vid_s["frames"][i], "SDF", vid_s["fphase"][i])
        sep = np.full((left.shape[0], 4, 3), 255, np.uint8)
        clip.append(np.hstack([left, sep, right]))
    else:
        clip.append(left)

VIDEO_PATH = "threading_tighten.mp4"
imageio.mimsave(VIDEO_PATH, clip, fps=30)
print(f"saved {VIDEO_PATH}  ({len(clip)} frames @ 30 fps)")
Video(VIDEO_PATH, embed=True, width=700)

## 6 · Compute-cost benchmark — what does fidelity cost?

Faithful contact is not free. Here we measure the wall-clock cost of `mj_step`
in each regime: the SDF thread evaluates the helix distance/gradient and resolves
a stiffer contact every step. We report **ms per step**, **steps/second**, and the
**slowdown factor** — the trade-off curve at the heart of AutoBio.

In [ ]:
# ── Time mj_step in each regime ──
import time

def bench_steps(xml, n_steps=4000, warmup=200):
    model = make_model(xml); data = mujoco.MjData(model)
    for _ in range(warmup):
        data.ctrl[0] = 1.0; mujoco.mj_step(model, data)
    t0 = time.perf_counter()
    for _ in range(n_steps):
        data.ctrl[0] = 1.0; mujoco.mj_step(model, data)
    dt = (time.perf_counter() - t0) / n_steps
    return dt * 1e3, 1.0 / dt        # ms/step, steps/s

ms_n, sps_n = bench_steps(naive_scene_xml())
print(f"NAÏVE : {ms_n:.4f} ms/step | {sps_n:,.0f} steps/s")
if PLUGIN_OK:
    ms_s, sps_s = bench_steps(sdf_scene_xml())
    print(f"SDF   : {ms_s:.4f} ms/step | {sps_s:,.0f} steps/s")
    print(f"Slowdown from faithful thread contact: {ms_s/ms_n:.2f}x")

In [ ]:
# ── Bar chart of the compute trade-off ──
labels = ["naïve\n(kinematic)"]; vals = [ms_n]
if PLUGIN_OK: labels.append("SDF plugin\n(faithful)"); vals.append(ms_s)
plt.figure(figsize=(5,4))
bars = plt.bar(labels, vals, color=["#888", "#d9534f"][:len(vals)])
plt.ylabel("ms per mj_step (lower = faster)")
plt.title("Compute cost of physical fidelity")
for b, v in zip(bars, vals):
    plt.text(b.get_x()+b.get_width()/2, v, f"{v:.3f}", ha="center", va="bottom")
plt.show()

## 7 · Does fidelity help a learned policy? A tiny VLA on each regime

Now the payoff. We collect **image + proprioception + language** demonstrations
of the screwing task in *each* simulator, train an identical tiny
Vision-Language-Action policy on each demo set, and then evaluate **both
policies in the faithful (SDF) environment** — the one that behaves like reality.

**The hypothesis (AutoBio's thesis):** the policy trained on physics-faithful
demonstrations transfers to the faithful task better than the one trained on
naïve demonstrations, because the naïve simulator never exposed the real
rotation→descent contact relationship or the tight terminal state.

> ⚠️ **Didactic stand-in.** This `TinyVLA` is a ~0.3 M-parameter CNN, *not* π₀/RDT.
> It captures the V-L-A structure (image encoder + instruction embedding + proprio
> → action chunk) so the experiment fits on a T4. If the plugin is unavailable
> this whole section is skipped.

In [ ]:
# ── Demonstration collection ──
# Observation: OBS_HW x OBS_HW RGB + proprio [hinge, slide]; instruction is a fixed token.
# Action: next-step hinge delta (what the expert commands). The cap descends
# according to whichever physics is active, so demos encode that physics.
#
# Performance: we build the model ONCE per scene, reuse a single cached renderer,
# and reset MjData between demos (mj_resetData) — NOT rebuild the model each demo.
# Caps + a NaN guard + progress prints guarantee the cell terminates.
import time

INSTRUCTION  = "screw the cap onto the tube until tight"
OBS_HW       = 64
TARGET_DEPTH = 0.006     # ~6 mm of descent counts as "tight" (= 2 thread turns)
DELTA        = 0.12      # expert hinge increment per macro-step (rad)
SUBSTEPS     = 8         # sim steps per macro-step
MAX_MACRO    = 160       # hard cap on macro-steps per demo

def collect_demos(xml, n_demos=6, seed0=0, tag=""):
    """Roll out the expert (steady screwing) and record (img, proprio, action)."""
    model = make_model(xml)                       # built ONCE for this scene
    data  = mujoco.MjData(model)
    h_adr, s_adr = jid(model, "hinge"), jid(model, "slide")
    obs_img, obs_prop, acts = [], [], []
    t0 = time.perf_counter()
    for d in range(n_demos):
        rng = np.random.default_rng(seed0 + d)
        mujoco.mj_resetData(model, data)          # reset instead of rebuilding
        start = float(rng.uniform(0, 0.3)); data.qpos[h_adr] = start
        mujoco.mj_forward(model, data)
        ang = start
        for _ in range(MAX_MACRO):
            img  = render_rgb(model, data, OBS_HW, OBS_HW)
            prop = np.array([data.qpos[h_adr], data.qpos[s_adr]], np.float32)
            obs_img.append(img); obs_prop.append(prop); acts.append([DELTA])
            ang += DELTA; data.ctrl[0] = ang
            for _ in range(SUBSTEPS):
                mujoco.mj_step(model, data)
            if not np.isfinite(data.qpos).all():          # instability guard
                print(f"  [{tag}] demo {d}: non-finite state, ending early"); break
            if -data.qpos[s_adr] >= TARGET_DEPTH:         # reached "tight"
                break
        print(f"  [{tag}] demo {d+1}/{n_demos} done "
              f"(depth={-data.qpos[s_adr]*1000:.1f} mm, {len(obs_img)} frames so far, "
              f"{time.perf_counter()-t0:.1f}s)")
    X_img = np.asarray(obs_img, np.float32).transpose(0, 3, 1, 2) / 255.0
    return (torch.tensor(X_img),
            torch.tensor(np.asarray(obs_prop, np.float32)),
            torch.tensor(np.asarray(acts, np.float32)))

if PLUGIN_OK:
    print("Collecting NAÏVE demos…");  naive_demos = collect_demos(naive_scene_xml(), 6, 0,   "naïve")
    print("Collecting SDF demos…");    sdf_demos   = collect_demos(sdf_scene_xml(),   6, 100, "sdf")
    print("naïve demo frames:", naive_demos[0].shape[0],
          "| sdf demo frames:", sdf_demos[0].shape[0])
else:
    print("Plugin unavailable — skipping the VLA experiment.")

In [ ]:
# ── TinyVLA: a minimal vision-language-action behaviour-cloning policy ──
class TinyVLA(nn.Module):
    """CNN vision encoder + instruction embedding + proprio MLP -> action.

    A deliberately small stand-in for a real VLA so it trains in minutes on a T4.
    The single fixed instruction is embedded via a lookup, mirroring how a real
    VLA conditions on language.
    """
    def __init__(self, n_instr=1, act_dim=1):
        super().__init__()
        self.vision = nn.Sequential(
            nn.Conv2d(3, 16, 5, 2, 2), nn.ReLU(),
            nn.Conv2d(16, 32, 3, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, 2, 1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten())          # -> 32
        self.instr = nn.Embedding(n_instr, 16)
        self.head = nn.Sequential(
            nn.Linear(32 + 16 + 2, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, act_dim))

    def forward(self, img, prop, instr_id):
        v = self.vision(img)
        e = self.instr(instr_id)
        return self.head(torch.cat([v, e, prop], dim=-1))

def train_policy(demos, epochs=25, bs=64, lr=1e-3, tag=""):
    img, prop, act = (t.to(DEVICE) for t in demos)
    instr = torch.zeros(len(img), dtype=torch.long, device=DEVICE)
    net = TinyVLA().to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    lossf = nn.MSELoss()
    n = len(img)
    for ep in range(epochs):
        perm = torch.randperm(n, device=DEVICE); tot = 0.0
        for i in range(0, n, bs):
            idx = perm[i:i+bs]
            pred = net(img[idx], prop[idx], instr[idx])
            loss = lossf(pred, act[idx])
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item()*len(idx)
        if ep % 5 == 0 or ep == epochs-1:
            print(f"[{tag}] epoch {ep:02d}  loss {tot/n:.5f}")
    return net.eval()

if PLUGIN_OK:
    print("Training policy on NAÏVE demos:");  net_naive = train_policy(naive_demos, tag="naïve")
    print("Training policy on SDF demos:");    net_sdf   = train_policy(sdf_demos,   tag="sdf")

In [ ]:
# ── Evaluate BOTH policies in the FAITHFUL (SDF) environment ──
# Same performance rules: build the SDF model ONCE, reuse the cached renderer,
# reset MjData per episode. Each rollout is hard-capped at MAX_MACRO steps.
@torch.no_grad()
def evaluate(net, n_episodes=10, seed0=500, tag=""):
    """Roll the policy out in the SDF env; success = reach target depth."""
    model = make_model(sdf_scene_xml())           # built ONCE
    data  = mujoco.MjData(model)
    h_adr, s_adr = jid(model, "hinge"), jid(model, "slide")
    succ = 0
    for e in range(n_episodes):
        rng = np.random.default_rng(seed0 + e)
        mujoco.mj_resetData(model, data)
        data.qpos[h_adr] = float(rng.uniform(0, 0.3)); mujoco.mj_forward(model, data)
        ang = data.qpos[h_adr]
        for _ in range(MAX_MACRO):
            img = render_rgb(model, data, OBS_HW, OBS_HW)
            x = torch.tensor(img.transpose(2,0,1)[None]/255., dtype=torch.float32, device=DEVICE)
            prop = torch.tensor([[data.qpos[h_adr], data.qpos[s_adr]]],
                                dtype=torch.float32, device=DEVICE)
            instr = torch.zeros(1, dtype=torch.long, device=DEVICE)
            delta = float(np.clip(net(x, prop, instr).item(), -0.15, 0.15))
            ang += delta; data.ctrl[0] = ang
            for _ in range(SUBSTEPS):
                mujoco.mj_step(model, data)
            if not np.isfinite(data.qpos).all():
                break
            if -data.qpos[s_adr] >= TARGET_DEPTH:
                succ += 1; break
    print(f"  [{tag}] {succ}/{n_episodes} episodes reached target")
    return succ / n_episodes

if PLUGIN_OK:
    print("Evaluating policy trained on NAÏVE demos:"); sr_naive = evaluate(net_naive, tag="naïve")
    print("Evaluating policy trained on SDF demos:");   sr_sdf   = evaluate(net_sdf,   tag="sdf")
    print(f"Success in the FAITHFUL env  | trained-on-naïve: {sr_naive:.0%} "
          f"| trained-on-SDF: {sr_sdf:.0%}")

In [ ]:
# ── The headline plot ──
if PLUGIN_OK:
    plt.figure(figsize=(5,4))
    bars = plt.bar(["trained on\nnaïve demos", "trained on\nSDF demos"],
                   [sr_naive, sr_sdf], color=["#888", "#5cb85c"])
    plt.ylabel("success rate in faithful task")
    plt.ylim(0, 1.05)
    plt.title("Lab-specific physics → better policy")
    for b, v in zip(bars, [sr_naive, sr_sdf]):
        plt.text(b.get_x()+b.get_width()/2, v, f"{v:.0%}", ha="center", va="bottom")
    plt.show()
    print("Takeaway: demonstrations from the physics-faithful simulator yield a "
          "policy that performs better on the real (faithful) screwing task — "
          "AutoBio's central argument, on a single T4.")

## 8 · Bonus & discussion questions

### Questions to think about
1. **Pitch & gauge.** Re-run §6 with `THREAD["pitch"]` halved or `bolt_gauge`
   changed. How does thread fineness affect contact stiffness, the timestep you
   need for stability, and the per-step cost?
2. **Why does the naïve cap unscrew?** Trace the equality constraint in §4: with
   no contact friction, what restoring effect (if any) opposes gravity? What
   would you have to *script* to fake self-locking, and why is that brittle?
3. **Fidelity vs. cost.** From §6, at what simulation scale (number of threaded
   parts, parallel envs) would the SDF overhead dominate data-collection time?
4. **Generalisation.** Our `TinyVLA` sees one fixed instruction. How would the
   experiment change with multiple tasks/instructions, and would faithful physics
   matter *more* or *less*?

### Going further
* **Swap the policy.** Replace `TinyVLA` with an ACT or diffusion-policy head, or
  a small LeRobot `SmolVLA`, and re-run the A/B comparison.
* **Use the full AutoBio task.** Run the real bimanual `screw_tighten.py`
  (Aloha + analytical IK) and render with the LeRobot-format pipeline; train π₀
  or RDT on an 80 GB GPU as in the paper.
* **Other AutoBio plugins.** The same `.so` registers `mjlab.passive.detent`
  (click stops) and `mjlab.sdf.grid`. Try the detent mechanism on an instrument dial.
* **Photoreal rendering.** Switch MuJoCo's OpenGL renderer for AutoBio's Blender
  Cycles backend (~100× slower) and study the sim-to-real rendering gap.

---
*Built as a didactic reconstruction of **AutoBio: A Simulation and Benchmark for
Robotic Automation in Digital Biology Laboratory** (Lan et al., arXiv:2505.14030).
Code/assets: github.com/autobio-bench/AutoBio. For research & education only.*